# Notebook 16 – Complete Model Optimization Project

**Workflow:** Data → Features → Baseline → Multiple Models → Evaluation → Tuning → Optimization → Error Analysis → Final Model

**Dataset:** Online Retail transactions (`data.csv`)

**Business problem:** Flag **international orders** (non-UK) so the team can prepare export paperwork and shipping. International orders are only ~11% of the data, so this is an **imbalanced** problem.

**Metrics chosen:** Accuracy is misleading here (always saying "UK" scores ~89%). We use **ROC-AUC** to compare models and **Precision / Recall / F1** (for the international class) for the final decision.

## 1. Data
Load the data, remove returns/errors (non-positive quantity or price), and take an 8,000-row sample for speed.

In [ ]:
import pandas as pd, numpy as np, time
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, RandomizedSearchCV, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score,
                             f1_score, confusion_matrix, precision_recall_curve)
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(8000, random_state=42)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.shape

## 2. Features
Create `TotalPrice` and time features (`Hour`, `DayOfWeek`, `Month`) — international customers may order at different times than UK ones. The target is `1` = international.

In [ ]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['Month'] = df['InvoiceDate'].dt.month
features = ['Quantity', 'UnitPrice', 'TotalPrice', 'Hour', 'DayOfWeek', 'Month']
X = df[features]
y = (df['Country'] != 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("International share:", round(y.mean(), 3))

## 3. Baseline
A model that always predicts the majority class. Every real model must beat this on the metrics that matter.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
pred = baseline.predict(X_test)
print("Baseline accuracy:", round((pred == y_test).mean(), 3))
print("Baseline F1:      ", f1_score(y_test, pred, zero_division=0))
print("Baseline ROC-AUC: ", roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1]))

## 4. Multiple Models
Train several different models. The unlimited Decision Tree is included on purpose to show overfitting.

In [ ]:
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Decision Tree (no limit)': DecisionTreeClassifier(random_state=42),
    'Decision Tree (depth 5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}
for m in models.values():
    m.fit(X_train, y_train)
print("Trained:", list(models))

## 5. Evaluation
Compare **train AUC**, **cross-validated AUC** (on training data) and **test AUC**. A big gap between train and CV/test means overfitting.

In [ ]:
rows = []
for name, m in models.items():
    cv_auc = cross_val_score(m, X_train, y_train, cv=skf, scoring='roc_auc').mean()
    rows.append({
        'Model': name,
        'Train AUC': roc_auc_score(y_train, m.predict_proba(X_train)[:, 1]),
        'CV AUC': cv_auc,
        'Test AUC': roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]),
    })
results = pd.DataFrame(rows).round(3)
results['Train-CV Gap'] = (results['Train AUC'] - results['CV AUC']).round(3)
results.sort_values('CV AUC', ascending=False)

**Reading the table:**
- The unlimited Decision Tree has a train AUC of 1.0 but a much lower CV/test AUC → **overfitting**.
- Logistic Regression scores low everywhere → **underfitting** (too simple for these features).
- Random Forest and Gradient Boosting have the best CV AUC. Gradient Boosting has a smaller train-CV gap, so it generalizes better.

We tune **Gradient Boosting** next.

## 6. Tuning
Use `RandomizedSearchCV` (faster than a full grid) to search key Gradient Boosting hyperparameters, scored by ROC-AUC.

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.7, 0.85, 1.0],
    'min_samples_leaf': [5, 20, 50],
}
search = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), param_dist,
                            n_iter=8, cv=3, scoring='roc_auc', random_state=42)
search.fit(X_train, y_train)
tuned = search.best_estimator_
print("Best params:", search.best_params_)
print("Best CV AUC:", round(search.best_score_, 3))

In [ ]:
default_gb = models['Gradient Boosting']
print("Default GB — Train AUC:", round(roc_auc_score(y_train, default_gb.predict_proba(X_train)[:, 1]), 3),
      "| Test AUC:", round(roc_auc_score(y_test, default_gb.predict_proba(X_test)[:, 1]), 3))
print("Tuned GB   — Train AUC:", round(roc_auc_score(y_train, tuned.predict_proba(X_train)[:, 1]), 3),
      "| Test AUC:", round(roc_auc_score(y_test, tuned.predict_proba(X_test)[:, 1]), 3))

Tuning gives only a **small gain in AUC**, but the tuned model is simpler (shallower trees, subsampling), so its train-test gap is smaller — a more stable model. The big improvement comes from the next step.

## 7. Optimization – Threshold Tuning
At the default threshold (0.5) the model almost never predicts "international" because that class is rare, so **recall is very low**. We pick a better threshold using **cross-validated predictions on the training data** (never the test set), choosing the threshold that maximizes F1.

In [ ]:
oof_probs = cross_val_predict(tuned, X_train, y_train, cv=skf, method='predict_proba')[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof_probs)
f1_curve = 2 * prec * rec / (prec + rec + 1e-9)
best_threshold = thr[f1_curve[:-1].argmax()]
print("Best threshold (max F1 on training CV):", round(best_threshold, 3))

In [ ]:
import matplotlib.pyplot as plt
plt.plot(thr, prec[:-1], label='Precision')
plt.plot(thr, rec[:-1], label='Recall')
plt.plot(thr, f1_curve[:-1], label='F1')
plt.axvline(best_threshold, color='gray', linestyle='--', label='Chosen threshold')
plt.xlabel('Threshold'); plt.ylabel('Score')
plt.title('Threshold Tuning (training CV predictions)')
plt.legend(); plt.show()

## 8. Error Analysis
Apply the chosen threshold to the test set and look at *where* the model makes mistakes.

In [ ]:
test_probs = tuned.predict_proba(X_test)[:, 1]
final_pred = (test_probs >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, final_pred).ravel()
print(f"True Negatives={tn}, False Positives={fp}, False Negatives={fn}, True Positives={tp}")

In [ ]:
err = X_test.copy()
err['y'] = y_test.values
err['pred'] = final_pred
err['Country'] = df.loc[X_test.index, 'Country']
intl = err[err['y'] == 1]
by_country = intl.groupby('Country')['pred'].agg(orders='count', recall='mean')
by_country.sort_values('orders', ascending=False).head(6).round(2)

In [ ]:
err['QuantityBucket'] = pd.qcut(err['Quantity'], 4, duplicates='drop')
(err['y'] != err['pred']).groupby(err['QuantityBucket'], observed=True).mean().round(3)

**What the errors tell us:**
- The model **catches about 58% of international orders** but raises many false alarms (about 400 false positives) — precision is low.
- **Country matters:** Netherlands, France and Germany are caught fairly well, while smaller countries like Spain are missed completely.
- **Order size matters:** errors are rare for small orders (they are almost always predicted UK) but reach ~50% for larger orders, where the lowered threshold flags many big UK orders as international.
- Order size, price and time alone are only weakly linked to country, so the features are the main limit, not the algorithm.
- **Next improvement:** add richer features (customer history, product/`StockCode` patterns) rather than tuning further.

## 9. Final Model
Compare every stage on the **test set** using the metrics that matter for this problem.

In [ ]:
def report(name, probs, threshold=0.5):
    pred = (probs >= threshold).astype(int)
    return {
        'Model': name,
        'Accuracy': round((pred == y_test).mean(), 3),
        'Precision': round(precision_score(y_test, pred, zero_division=0), 3),
        'Recall': round(recall_score(y_test, pred, zero_division=0), 3),
        'F1': round(f1_score(y_test, pred, zero_division=0), 3),
        'ROC-AUC': round(roc_auc_score(y_test, probs), 3),
    }
final_table = pd.DataFrame([
    report('Baseline', baseline.predict_proba(X_test)[:, 1]),
    report('Default Gradient Boosting (thr 0.5)', default_gb.predict_proba(X_test)[:, 1]),
    report('Tuned Gradient Boosting (thr 0.5)', test_probs),
    report('Tuned GB + tuned threshold (FINAL)', test_probs, best_threshold),
])
final_table

### Why the final model was selected

**Final model: Tuned Gradient Boosting + optimized threshold.**

1. **Best ranking ability.** Gradient Boosting (default and tuned) has the highest cross-validated ROC-AUC, ahead of Logistic Regression and single trees.
2. **Good generalization.** The unlimited Decision Tree overfits (train AUC 1.0, much lower on test). The tuned Gradient Boosting has a small train-test gap, so we can trust its test score.
3. **Accuracy was not the deciding factor.** The baseline already has ~89% accuracy but finds **zero** international orders. Our final model has clearly lower accuracy (~70%) but finds most of them (recall ≈ 0.58) — which is the actual business goal.
4. **The threshold made the biggest difference.** Moving from 0.5 to the tuned threshold lifts recall and F1 sharply, at the cost of precision. This trade-off is acceptable because missing an international order (no export paperwork) is usually costlier than a false alarm.
5. **Reasonable complexity.** It trains in seconds and is more accurate than simpler models; a single tree is easier to explain but overfits.

**Limitation:** performance is modest (AUC ≈ 0.75) — better features are the best next step.